# Notebook 3. Dữ liệu chuyên gia và cấp 5 (mạng policy học có giám sát)

Người phụ trách: Trần Quỳnh Hương (cấp 5), sinh dữ liệu và hàm mạng: Nguyễn Trọng Phúc.

Nội dung:
1. Sinh ván chuyên gia trên nhiều máy (mỗi phần một seed), rồi gộp.
2. Thống kê dữ liệu: số ván, tỷ lệ thắng/hòa, độ dài ván.
3. Gán nhãn mọi thế cờ bằng Alpha-Beta tìm sâu (phiên bản 2).
4. Tạo tập train/val/test chia theo ván, loại thế trùng; huấn luyện mạng policy, đường cong học, độ chính xác trên val/test.
5. Đánh giá cấp 5: đấu cấp 1–3 và độ chính xác chiến thuật.

**Phiên bản 1** (tag git `v1-ket-qua`): nhãn là nước đi của bên THẮNG trong ván chuyên gia (50% Alpha-Beta độ sâu 3, 50% MCTS 1000 mô phỏng), giống chương 11 của sách. Kết quả: test acc 0,505, thấy nước thắng ngay chỉ 14% số lần, thua cấp 2 (score 0,10). Nguyên nhân: nhãn nhiễu (hai chuyên gia khác nhau chọn khác nhau ở cùng thế cờ, MCTS có ngẫu nhiên) và chỉ một nửa số thế có nhãn.

**Phiên bản 2**: giữ nguyên 3000 ván, nhưng **mọi** thế cờ (cả hai bên, bỏ thế trùng) được gán nhãn bằng một chuyên gia duy nhất, mạnh hơn: Alpha-Beta của cấp 3 tìm sâu `data.label_depth` (5, tức nhìn trước 6 nước, cấp 3 chỉ nhìn 4). Nhãn nhất quán, luôn thấy nước thắng ngay và nước phải chặn.
Nguy cơ: mạng học theo đúng điểm yếu của hàm lượng giá Alpha-Beta.

Đầu ra: `files/policy_conn.keras` (đẩy lên git), `files/games_*.p`, `files/policy_data.npz` (không đẩy, tạo lại được).

In [ ]:
# QUICK = True: chạy thử vài ván, vài epoch để kiểm tra code (ghi vào thư mục con thu_nhanh/)
# QUICK = False: chạy thật theo config.yaml, kết quả dùng cho báo cáo
QUICK = False

import os
if not os.path.exists("config.yaml"):
    raise RuntimeError("Hãy mở Jupyter trong thư mục final_connect_four (nơi có config.yaml).")

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.experiments import setup, summaries_table, with_random_opening, move_times_of
from utils.arena import run_match, format_summary, read_log, first_player_stats
from utils.data import machine_info
import keras
from utils import data, nets, policy_net, tactics
ex = setup(QUICK)
cfg = ex.cfg
print("Cấu hình máy:", machine_info())

## 1. Sinh ván chuyên gia

Ba phần, mỗi phần 1000 ván (`data.games_per_machine`), seed 1, 2, 3 lấy từ `data.seeds`. Tên phần chỉ là nhãn của seed.
Phần nào đã có file đủ số ván thì ô dưới bỏ qua. Mỗi 50 ván lưu một lần nên bị ngắt vẫn giữ được phần đã sinh (nhưng chạy lại sẽ sinh lại phần đó từ đầu).

Lần chạy thật: cả ba phần chạy song song trên cùng một máy (Windows 11, 16 luồng). Phần `phuc` chạy bằng ô dưới; phần `huong` và `thienphuc` chạy bằng dòng lệnh, cùng hàm và tham số:

`python -c "from utils import data; data.generate_games(1000, seed=2, out_path='files/games_huong.p')"`

`python -c "from utils import data; data.generate_games(1000, seed=3, out_path='files/games_thienphuc.p')"`

Thời gian và cấu hình máy của từng phần nằm trong các file `files/games_*.p.json`, tổng hợp ở bảng mục 2.

In [ ]:
PARTS_TO_RUN = ["phuc", "huong", "thienphuc"]

data_cfg = cfg["data"]
for part in PARTS_TO_RUN:
    out_path = ex.model(f"games_{part}.p")
    if os.path.exists(out_path + ".json"):
        with open(out_path + ".json", encoding="utf-8") as f:
            meta = json.load(f)
        if meta.get("n_games_done") == meta.get("n_games_target"):
            print("Phần", part, "đã xong, bỏ qua")
            continue
    print("Sinh phần", part, "với seed", data_cfg["seeds"][part])
    data.generate_games(ex.n(data_cfg["games_per_machine"], 6), seed=data_cfg["seeds"][part], out_path=out_path,
                        minimax_depth=data_cfg["minimax_depth"], mcts_simulations=data_cfg["mcts_simulations"])

## 2. Gộp và thống kê dữ liệu

Ô này cần đủ các file `games_*.p` của mọi phần trong `data.seeds`. Thiếu phần nào thì ô sẽ báo.

In [ ]:
paths = []
missing = []
for part in data_cfg["seeds"]:
    path = ex.model(f"games_{part}.p")
    if os.path.exists(path):
        paths.append(path)
    else:
        missing.append(part)
if missing:
    print("CẢNH BÁO: còn thiếu phần", missing, "- kết quả dưới đây chỉ dùng các phần đã có")
games = data.merge_game_files(paths)

# thời gian sinh và cấu hình máy của từng phần (đề yêu cầu khai báo tài nguyên)
rows = []
for path in paths:
    with open(path + ".json", encoding="utf-8") as f:
        meta = json.load(f)
    rows.append({"file": os.path.basename(path), "seed": meta["seed"], "số ván": meta["n_games_done"],
                 "giây": meta["seconds"], "CPU": meta["machine"]["processor"],
                 "số nhân": meta["machine"]["cpu_count"], "hệ điều hành": meta["machine"]["system"]})
gen_table = pd.DataFrame(rows)
gen_table.to_csv(ex.table("nb3_sinh_du_lieu.csv"), index=False, encoding="utf-8-sig")
gen_table

In [ ]:
red = 0
yellow = 0
draw = 0
lengths = []
for game in games:
    if game["reward"] == 1:
        red += 1
    elif game["reward"] == -1:
        yellow += 1
    else:
        draw += 1
    lengths.append(len(game["history"]))
n = len(games)
print(f"{n} ván | đỏ thắng {red / n:.3f}, vàng thắng {yellow / n:.3f}, hòa {draw / n:.3f}")
print(f"độ dài ván: trung bình {np.mean(lengths):.1f} nước, ngắn nhất {min(lengths)}, dài nhất {max(lengths)}")

## 3. Gán nhãn bằng Alpha-Beta tìm sâu

Chạy song song nhiều tiến trình (mặc định số nhân CPU trừ 2). Cứ 5000 thế lưu một lần vào `files/nhan_alphabeta.p`; bị ngắt thì chạy lại ô này, nó tiếp tục từ chỗ đã lưu. Thời gian và cấu hình máy ghi trong `files/nhan_alphabeta.p.json`.

In [ ]:
label_depth = ex.n(cfg["data"]["label_depth"], 3)
positions = data.game_positions(games)
if QUICK:
    positions = positions[:300]
print("Số thế cờ cần gán nhãn (đã bỏ thế trùng):", len(positions))
labeled = data.label_positions(positions, depth=label_depth, processes=cfg["data"]["label_processes"],
                               out_path=ex.model("nhan_alphabeta.p"))

with open(ex.model("nhan_alphabeta.p") + ".json", encoding="utf-8") as f:
    label_meta = json.load(f)
proven_win = 0
proven_loss = 0
for game_id, moves, label, best in labeled:
    if best >= 1.0:
        proven_win += 1
    elif best <= -1.0:
        proven_loss += 1
print(f"Gán nhãn {len(labeled)} thế, độ sâu {label_depth}, {label_meta['processes']} tiến trình, {label_meta['seconds']} giây")
print(f"Tìm kiếm chứng minh được: người sắp đi thắng chắc {proven_win} thế, thua chắc {proven_loss} thế")

## 4. Tập dữ liệu và huấn luyện mạng policy

Mẫu = (thế cờ nhìn từ người sắp đi, cột tốt nhất theo Alpha-Beta tìm sâu). Chia train/val/test = 80/10/10 **theo ván**, loại cặp trùng trong từng tập và loại khỏi val/test các thế đã có trong train.
Tập train được thêm bản lật gương (Connect Four đối xứng trái phải: cột c thành cột 8 − c), val và test giữ nguyên.

In [ ]:
policy_set = data.make_dataset(games, kind="search_policy", seed=0, mirror=cfg["train_policy"]["mirror"],
                               labeled=labeled)
print(data.describe_dataset(policy_set))

counts = np.bincount(policy_set["y_train"], minlength=7)
label_table = pd.DataFrame({"cột": range(1, 8), "số mẫu train": counts,
                            "tỷ lệ": np.round(counts / counts.sum(), 3)})
# mốc so sánh: luôn đoán cột xuất hiện nhiều nhất trong train
majority = int(np.argmax(counts))
baseline = float(np.mean(policy_set["y_test"] == majority))
print(f"Mốc 'luôn đoán cột {majority + 1}': độ chính xác trên test = {baseline:.3f}")
label_table

Kiến trúc của chương 11 sách (Conv 128 bộ lọc 4×4, Dense 64, Dense 64, Softmax 7), Adam, sparse categorical cross-entropy.
Siêu tham số lấy từ `train_policy` trong `config.yaml`. Dừng sớm khi val_loss không giảm trong `patience` epoch liền và giữ trọng số tốt nhất (lần đầu của phiên bản 1 không dừng sớm và học thuộc từ epoch 6). Tập test chỉ dùng một lần ở cuối.

In [ ]:
train_cfg = cfg["train_policy"]
keras.utils.set_random_seed(0)
policy_path = ex.model("policy_conn.keras")
model, report = policy_net.train_policy(games, path=policy_path, epochs=ex.n(train_cfg["epochs"], 2),
                                        batch_size=train_cfg["batch_size"],
                                        learning_rate=train_cfg["learning_rate"], seed=0, verbose=2,
                                        mirror=train_cfg["mirror"], patience=train_cfg["patience"],
                                        labeled=labeled)
print(f"Chạy {report['epochs_run']} epoch, giữ trọng số epoch {report['best_epoch']} (val_loss thấp nhất)")
print(f"Huấn luyện {report['train_seconds']} giây | val acc {report['val_accuracy']:.3f} | "
      f"test acc {report['test_accuracy']:.3f} (mốc luôn đoán một cột: {baseline:.3f})")
print("Số tham số của mạng:", model.count_params())

In [ ]:
curve = report["curve"]
epochs = range(1, len(curve["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, curve["loss"], label="train")
axes[0].plot(epochs, curve["val_loss"], label="val")
axes[0].set_title("Loss")
axes[1].plot(epochs, curve["accuracy"], label="train")
axes[1].plot(epochs, curve["val_accuracy"], label="val")
axes[1].axhline(baseline, color="#999999", linestyle="--", label="luôn đoán một cột")
axes[1].set_title("Độ chính xác")
for ax in axes:
    ax.axvline(report["best_epoch"], color="#999999", linestyle=":", linewidth=1)
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.25)
    ax.legend()
fig.suptitle(f"Đường cong học của mạng policy (cấp 5), giữ epoch {report['best_epoch']}")
fig.tight_layout()
fig.savefig(ex.fig("nb3_duong_cong_policy.png"), dpi=150)

## 5. Đánh giá cấp 5

Cấp 5 luôn đi cột có xác suất cao nhất nên là tác tử tất định; mỗi ván mở đầu bằng 2 nước ngẫu nhiên có seed để các ván khác nhau.
Đây là kiểm tra nhanh sau huấn luyện; số liệu chính thức của cấp 5 lấy ở giải đấu nb6.

In [ ]:
n_games = ex.n(100, 6)
opening = cfg["tournament"]["opening_random_moves"]
log_path = ex.log("nb3_danh_gia_cap5.jsonl")
cap5 = with_random_opening(ex.agent("cap5"), opening)

summaries = []
for k, name in enumerate(["cap1", "cap2", "cap3"]):
    s = run_match(cap5, with_random_opening(ex.agent(name), opening), "cap5", name, n_games,
                  seed=600 + k, log_path=log_path)
    summaries.append(s)
    print(format_summary(s))
cap5_table = summaries_table(summaries)
cap5_table.to_csv(ex.table("nb3_danh_gia_cap5.csv"), index=False, encoding="utf-8-sig")
cap5_table

In [ ]:
tactics_path = ex.model("tactics.json")
if os.path.exists(tactics_path):
    positions = tactics.load_tactics(tactics_path)
    move_fn, info_fn = ex.agent("cap5")
    print("Độ chính xác chiến thuật của cấp 5:", tactics.score_agent(move_fn, positions))
else:
    print("Chưa có", tactics_path, "- chạy mục 1 của nb2 (hoặc chép file từ máy kia) rồi chạy lại ô này.")